<a href="https://colab.research.google.com/github/jcjcjjc-kekeodt/CPE-311/blob/main/Hands_on_Activity_8_1_Aggregating_Pandas_DataFrames_DEBOLGADO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###**Hands-on Activity 8.1: Aggregating Data with Pandas** <br>
#8.1.1 Intended Learning Outcomes
After this activity, the student should be able to: <br>


*   Demonstrate querying and merging of dataframes
*   Perform advanced calculations on dataframes
*   Aggregate dataframes with pandas and numpy
*   Work with time series data





#8.1.2 Resources <br>


*   Computing Environment using Python 3.x
*   Attached Datasets (under Instructional Materials)



#8.1.3 Procedures <br>
The procedures can be found in the canvas module. Check the following under topics:
*   8.1 Weather Data Collection
*   8.2 Querying and Merging
*   8.3 Dataframe Operations
*   8.4 Aggregations
*   8.5 Time Series




#8.1.4 Data Analysis
Provide some comments here about the results of the procedures.

#8.1.5 Supplementary Activity

Using the CSV files provided and what we have learned so far in this module complete the following exercises:

1. With the earthquakes.csv file, select all the earthquakes in Japan with a magType of mb and a magnitude of 4.9 or greater.
2. Create bins for each full number of magnitude (for example, the first bin is 0-1, the second is 1-2, and so on) with a magType of ml and count how many are in each bin.
3. Using the faang.csv file, group by the ticker and resample to monthly frequency. Make the following aggregations:

 *   Mean of the opening price
 *   Maximum of the high price
 *   Minimum of the low price
 *   Mean of the closing price
 *   Sum of the volume traded

 4. Build a crosstab with the earthquake data between the tsunami column and the magType column. Rather than showing the frequency count, show the maximum
magnitude that was observed for each combination. Put the magType along the columns.
5. Calculate the rolling 60-day aggregations of OHLC data by ticker for the FAANG data. Use the same aggregations as exercise no. 3.
6. Create a pivot table of the FAANG data that compares the stocks. Put the ticker in the rows and show the averages of the OHLC and volume traded data.
7. Calculate the Z-scores for each numeric column of Netflix's data (ticker is NFLX) using apply().
8. Add event descriptions:
* Create a dataframe with the following three columns: ticker, date, and event. The columns should have the following values: <br>

- *ticker: 'FB'*
- *date: ['2018-07-25', '2018-03-19', '2018-03-20']*
- *event: ['Disappointing user growth announced after close.', 'Cambridge Analytica story', 'FTC investigation']*

* Set the index to ['date', 'ticker']
* Merge this data with the FAANG data using an outer join

9. Use the transform() method on the FAANG data to represent all the values in terms of the first date in the data. To do so, divide all the values for each ticker by the values
for the first date in the data for that ticker. This is referred to as an index, and the data for the first date is the base (https://ec.europa.eu/eurostat/statistics-explained/
index.php/ Beginners:Statisticalconcept-Indexandbaseyear). When data is in this format, we can easily see growth over time. Hint: transform() can take a function name.





In [18]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

eq = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/earthquakes.csv')
faang = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/faang.csv', parse_dates=['date'])


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
japan_mb = eq[
    (eq['parsed_place'] == 'Japan') &
    (eq['magType'] == 'mb') &
    (eq['mag'] >= 4.9)
]
print(japan_mb)


      mag magType           time                         place  tsunami  \
1563  4.9      mb  1538977532250  293km ESE of Iwo Jima, Japan        0   
2576  5.4      mb  1538697528010    37km E of Tomakomai, Japan        0   
3072  4.9      mb  1538579732490     15km ENE of Hasaki, Japan        0   
3632  4.9      mb  1538450871260    53km ESE of Hitachi, Japan        0   

     parsed_place  
1563        Japan  
2576        Japan  
3072        Japan  
3632        Japan  


In [20]:
ml_quakes = eq[eq['magType'] == 'ml']
ml_quakes['mag_bin'] = pd.cut(
    ml_quakes['mag'],
    bins=range(0, int(ml_quakes['mag'].max())+2)
)
bin_counts = ml_quakes['mag_bin'].value_counts().sort_index()
print(bin_counts)





mag_bin
(0, 1]    2207
(1, 2]    3105
(2, 3]     862
(3, 4]     122
(4, 5]       2
(5, 6]       1
Name: count, dtype: int64


/tmp/ipython-input-442/374693172.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ml_quakes['mag_bin'] = pd.cut(


In [21]:
faang['date'] = pd.to_datetime(faang['date'])
faang = faang.set_index('date')

monthly = faang.groupby('ticker').resample('ME').agg({
    'open':'mean',
    'high':'max',
    'low':'min',
    'close':'mean',
    'volume':'sum'
})
print(monthly.head())

crosstab = pd.crosstab(eq['tsunami'], eq['magType'], values=eq['mag'], aggfunc='max')
print(crosstab)

rolling = faang.groupby('ticker').rolling('60D').agg({
    'open':'mean',
    'high':'max',
    'low':'min',
    'close':'mean',
    'volume':'sum'
})
print(rolling.head())

pivot = faang.pivot_table(
    index='ticker',
    values=['open','high','low','close','volume'],
    aggfunc='mean'
)
print(pivot)



                         open      high       low       close     volume
ticker date                                                             
AAPL   2018-01-31  170.714690  176.6782  161.5708  170.699271  659679440
       2018-02-28  164.562753  177.9059  147.9865  164.921884  927894473
       2018-03-31  172.421381  180.7477  162.4660  171.878919  713727447
       2018-04-30  167.332895  176.2526  158.2207  167.286924  666360147
       2018-05-31  182.635582  187.9311  162.7911  183.207418  620976206
magType   mb  mb_lg    md   mh   ml  ms_20    mw  mwb  mwr  mww
tsunami                                                        
0        5.6    3.5  4.11  1.1  4.2    NaN  3.83  5.8  4.8  6.0
1        6.1    NaN   NaN  NaN  5.1    5.7  4.41  NaN  NaN  7.5
                         open      high       low       close       volume
ticker date                                                               
AAPL   2018-01-02  166.927100  169.0264  166.0442  168.987200   25555934.0
       2

In [23]:

nflx = faang[faang['ticker'] == 'NFLX']
z_scores = nflx.apply(
    lambda x: (x - x.mean()) / x.std() if np.issubdtype(x.dtype, np.number) else x
)
print(z_scores.head())

events = pd.DataFrame({
    'ticker':['FB']*3,
    'date':pd.to_datetime(['2018-07-25','2018-03-19','2018-03-20']),
    'event':['Disappointing user growth announced after close.',
             'Cambridge Analytica story',
             'FTC investigation']
}).set_index(['date','ticker'])

faang_reset = faang.reset_index()
faang_indexed = faang_reset.set_index(['date','ticker'])

merged = faang_indexed.merge(events, how='outer', left_index=True, right_index=True)
print(merged.head())

indexed = faang.groupby('ticker').transform(lambda x: x / x.iloc[0])
print(indexed.head())





           ticker      open      high       low     close    volume
date                                                               
2018-01-02   NFLX -2.500753 -2.516023 -2.410226 -2.416644 -0.088760
2018-01-03   NFLX -2.380291 -2.423180 -2.285793 -2.335286 -0.507606
2018-01-04   NFLX -2.296272 -2.406077 -2.234616 -2.323429 -0.959287
2018-01-05   NFLX -2.275014 -2.345607 -2.202087 -2.234303 -0.782331
2018-01-08   NFLX -2.218934 -2.295113 -2.143759 -2.192192 -1.038531
                        open       high        low      close    volume event
date       ticker                                                            
2018-01-02 AAPL     166.9271   169.0264   166.0442   168.9872  25555934   NaN
           AMZN    1172.0000  1190.0000  1170.5100  1189.0100   2694494   NaN
           FB       177.6800   181.5800   177.5500   181.4200  18151903   NaN
           GOOG    1048.3400  1066.9400  1045.2300  1065.0000   1237564   NaN
           NFLX     196.1000   201.6500   195.4200   201

#1. Earthquake Data (earthquakes.csv)<br>
* Queried all earthquakes in Japan with  and  using the  column.
*	Created magnitude bins (0–1, 1–2, …) for earthquakes with  and counted the number of events in each bin.
*	Built a crosstab between  and , showing the maximum magnitude observed for each combination. <br>

#2. FAANG Stock Data (faang.csv) |<br>
* Grouped by  and resampled to monthly frequency (), aggregating:
* Mean of opening price
* Maximum of high price
* Minimum of low price
* Mean of closing price
* Sum of volume traded
* Calculated rolling 60‑day aggregations of OHLCV data by ticker with the same aggregation functions.
* Created a pivot table comparing FAANG stocks, showing averages of OHLC and volume traded.
* Computed Z‑scores for each numeric column of Netflix () using .
#3. Event Data Integration
*	Built a DataFrame with , , and  columns for Facebook (FB) events.
*	Set the index to .
*	Merged this event DataFrame with FAANG data using an outer join.
#4. Index Transformation <br>
*	Used  to represent FAANG values relative to the first date for each ticker, creating an index format to visualize growth over time.

##**Conclusion** <br>
In this activity I learned how to query and filter datasets using conditions, how to categorize continuous values into bins, and how to aggregate data both by grouping and resampling. I practiced building crosstabs and pivot tables to compare categories, and applied rolling windows to smooth time series data. I also learned how to normalize values with Z‑scores, merge external event data with financial datasets, and transform stock prices into index form to highlight growth trajectories. These procedures strengthened my ability to handle real‑world datasets with Pandas, combining querying, aggregation, and time series analysis into a cohesive workflow.
